# SMS defect-type ordering probe

Use a **T4 GPU** and upload `sms_type_probe_bundle.zip`. Inference only: base `BAAI/bge-reranker-base` scores the fixed hybrid top-100 of 25 anchors. No labels are in the bundle.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, f'Upload exactly one bundle ZIP; received: {list(uploaded)}'
run_dir = Path('/content/sms_type_probe')
shutil.rmtree(run_dir, ignore_errors=True)
run_dir.mkdir(parents=True)
with zipfile.ZipFile(archives[0]) as zf:
    zf.extractall(run_dir)
%cd /content/sms_type_probe

In [ ]:
!pip -q install 'transformers>=4.44,<5' sentencepiece safetensors
import time; t0 = time.time()
!python evaluate_relation_reranker.py --data data --out output --batch-size 32 --max-length 256
print(f'{2500 / (time.time() - t0):.1f} pairs/s (includes model load)')

In [ ]:
!zip -qr sms_type_probe_result.zip output
from google.colab import files
files.download('sms_type_probe_result.zip')